# Greengard 2023 Table 3 & Table 4 (EFGP + Eigen top-q)

This notebook mirrors the "big example" setup from gp-shootout for Tables 3/4.
FLAM is removed; we sweep EFGP + Eigen top-q values similar to the Table 2 notebook.
Set `SIGMA_MODE` to `"table3"` (fixed sigma) or `"table4"` (sigma scales with N).

In [77]:
import sys
import time
import math
import csv
import importlib
from pathlib import Path

import numpy as np

root = Path.cwd()
parents = [root, root.parent, root.parent.parent]
for base in parents:
    if (base / "efgp_eigenpro_py").exists():
        sys.path.insert(0, str(base))
        break
else:
    raise RuntimeError("cannot find efgp_eigenpro_py from current path")

import efgp_eigenpro_py.efgp_solver as efgp_solver
import efgp_eigenpro_py.toeplitz as toeplitz
import efgp_eigenpro_py.eigenspace as eigenspace
importlib.reload(toeplitz)
importlib.reload(efgp_solver)
importlib.reload(eigenspace)

from efgp_eigenpro_py.efgp_solver import EFGPSolver
from efgp_eigenpro_py.kernels import make_matern
from efgp_eigenpro_py import benchmark as bm
importlib.reload(bm)

np.set_printoptions(precision=4, suppress=True)

In [ ]:
# Greengard 2023 big example settings
DIM = 2
WAVEVEC = np.array([3.0, 4.0], dtype=float)
PHASE = 1.3

VARIANCE = 1.0
LENGTHSCALE = 0.1
NU_MATERN = 1.5
KERNEL = make_matern(lengthscale=LENGTHSCALE, dim=DIM, nu=NU_MATERN, variance=VARIANCE)
KERNEL_LABEL = "Mat32"

SIGMA_BASE = 0.1
SIGMA_REF_N = 3_000_000

RUN_FULL_N = True
if RUN_FULL_N:
    N_LIST =  [3_000_000, 10_000_000, 30_000_000] #[3_000_000, 10_000_000] # [30_000_000]
    NTRGS_PER_D = 1000  # iNtrg=3 in the paper
else:
    N_LIST = [300_000, 1_000_000]
    NTRGS_PER_D = 200

EPS_LIST = [1e-5, 1e-7]
EPS_REF = 1e-8
COMPUTE_EEPM = False
EEPM_TRAIN_MAX_N = None  # set e.g. 2_000_000 to limit training EEPM
EEPM_TRAIN_TOPQ = None   # set list (e.g. [0, 8]) to limit training EEPM

TOP_Q_SWEEP = [0,32,48,64]
CG_MAXITER_BASELINE = 100000
CG_MAXITER_PRECOND = 50000
CG_MAXITER_REF = 200000

EIG_METHOD = "eigsh"
EIG_TOL = 1e-1
EIG_MAXITER = 50
EIG_BLOCK_SIZE = None
EIG_OVERSAMPLE = 2

L2_SCALED = True

TRAIN_CHUNK_SIZE = 2_000_000
MEAN_INCLUDE_TRAIN = False
SIGMA_MODE = "table3"  # "table3" or "table4"

ENABLE_SOLVE_BREAKDOWN = False

CSV_OUTPUT_DIR = Path("results")
CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH_TABLE3 = CSV_OUTPUT_DIR / "greengard_table3_breakdown.csv"
CSV_PATH_TABLE4 = CSV_OUTPUT_DIR / "greengard_table4_breakdown.csv"

BASE_SEED = 2026

print("RUN_FULL_N=", RUN_FULL_N, "N_LIST=", N_LIST, "NTRGS_PER_D=", NTRGS_PER_D)

RUN_FULL_N= True N_LIST= [3000000, 10000000, 30000000] NTRGS_PER_D= 1000


In [79]:
def f_eval(x: np.ndarray) -> np.ndarray:
    return np.cos(2.0 * math.pi * (x @ WAVEVEC) + PHASE)


def equispaced_grid(dim: int, n_per_d: int) -> np.ndarray:
    grid_1d = np.linspace(0.0, 1.0, n_per_d, endpoint=False)
    mesh = np.meshgrid(*([grid_1d] * dim), indexing="ij")
    return np.stack([g.reshape(-1) for g in mesh], axis=1)


def rms(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.sqrt(np.mean((a - b) ** 2)))


def sigma_for_N(n_value: int, mode: str) -> float:
    if mode == "table3":
        return float(SIGMA_BASE)
    if mode == "table4":
        return float(SIGMA_BASE * math.sqrt(n_value / SIGMA_REF_N))
    raise ValueError("unknown sigma mode: " + str(mode))


def seed_for_case(n_value: int, mode: str) -> int:
    if n_value in N_LIST:
        idx = N_LIST.index(n_value)
    else:
        idx = int(round(math.log10(n_value)))
    mode_offset = 0 if mode == "table3" else 100
    return int(BASE_SEED + mode_offset * 1000 + idx)


def make_targets(seed: int, sigma: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    y_noisy = Y_TRG_TRUE + sigma * rng.standard_normal(Y_TRG_TRUE.shape[0])
    return X_TRG, y_noisy, Y_TRG_TRUE


X_TRG = equispaced_grid(DIM, NTRGS_PER_D)
Y_TRG_TRUE = f_eval(X_TRG)

print("targets shape=", X_TRG.shape)

targets shape= (1000000, 2)


In [84]:
import numpy as np
import math

# 这会报错: AttributeError: module 'numpy' has no attribute 'math'
# print(np.math.gamma(5))

# 这会正常工作
print(math.gamma(5))

# numpy 自己的 gamma 函数是在 scipy.special 里 (如果您安装了 scipy)
# from scipy.special import gamma as np_gamma
print(np.math.gamma(5))

24.0
24.0


C:\Users\24681\AppData\Local\Temp\ipykernel_15552\787405056.py:12: DeprecationWarning: `np.math` is a deprecated alias for the standard library `math` module (Deprecated Numpy 1.25). Replace usages of `np.math` with `math`
  print(np.math.gamma(5))


In [80]:
def generate_train_chunks(
    dim: int,
    n_value: int,
    chunk_size: int,
    seed: int,
    sigma: float,
    *,
    return_mean: bool = False,
):
    rng = np.random.default_rng(seed)
    produced = 0
    while produced < n_value:
        m = min(chunk_size, n_value - produced)
        x = rng.uniform(0.0, 1.0, size=(m, dim))
        f = f_eval(x)
        y = f + sigma * rng.standard_normal(m)
        if return_mean:
            yield x, y, f
        else:
            yield x, y
        produced += m


def precompute_streaming_case(n_value: int, sigma: float, eps: float, seed: int):
    reg_lambda = sigma ** 2
    nufft_eps = eps / 10.0
    solver = EFGPSolver(
        KERNEL,
        reg_lambda=reg_lambda,
        eps=eps,
        nufft_tol=nufft_eps,
        l2scaled=L2_SCALED,
    )
    chunk_iter = generate_train_chunks(DIM, n_value, TRAIN_CHUNK_SIZE, seed, sigma)
    bounds = (np.zeros(DIM), np.ones(DIM))
    t0 = time.perf_counter()
    state = solver.precompute_streaming(chunk_iter, n_total=int(n_value), x_bounds=bounds)
    t1 = time.perf_counter()
    pre_times = {"time_precompute_total": t1 - t0}
    return solver, state, pre_times


def compute_train_eepm(
    solver: EFGPSolver,
    state,
    beta: np.ndarray,
    n_value: int,
    sigma: float,
    seed: int,
    *,
    max_n: int | None = None,
) -> float:
    n_eval = int(n_value) if max_n is None else int(min(n_value, max_n))
    if n_eval <= 0:
        return float("nan")
    remain = n_eval
    se_sum = 0.0
    count = 0
    for x_chunk, _y_chunk, f_chunk in generate_train_chunks(
        DIM,
        n_value,
        TRAIN_CHUNK_SIZE,
        seed,
        sigma,
        return_mean=True,
    ):
        if remain <= 0:
            break
        m = min(remain, x_chunk.shape[0])
        yhat = solver.predict(x_chunk[:m], beta, state)
        diff = yhat - f_chunk[:m]
        se_sum += float(np.sum(diff ** 2))
        count += int(m)
        remain -= int(m)
    return float(np.sqrt(se_sum / max(count, 1)))


def _solve_breakdown_header() -> str:
    return (
        "solve_s_total n_iter n_matvec t_matvec_total t_matvec_avg "
        "n_precond t_precond_total t_precond_avg t_cg_overhead"
    )


def print_table_header():
    header = (
        "sigma N d kernel eps top_q m pre_s eig_s precond_s solve_s mean_s total_s "
        "iters relres converged hit_max eepm_train eepm_new rms rmse_ex"
    )
    if ENABLE_SOLVE_BREAKDOWN:
        header = header + " " + _solve_breakdown_header()
    print(header)


def format_table_row(row: dict) -> str:
    converged = "Y" if row["converged"] else "N"
    hit_maxiter = "Y" if row["hit_maxiter"] else "N"
    base = (
        f"{row['sigma']:>8.2e} {row['N']:>10d} {row['dim']:>1d} {row['kernel']:>7s} {row['eps']:>8.1e} "
        f"{row['top_q']:>5d} {row['m']:>4d} {row['pre']:>8.3f} {row['eig']:>8.3f} "
        f"{row['precond']:>8.3f} {row['solve']:>8.3f} {row['mean']:>8.3f} {row['total']:>8.3f} "
        f"{row['iters']:>6d} {row['relres']:>8.2e} {converged:>9s} {hit_maxiter:>7s} "
        f"{row['eepm_train']:>9.2e} {row['eepm_new']:>9.2e} {row['rms']:>9.2e} {row['rmse_ex']:>9.2e}"
    )
    if not ENABLE_SOLVE_BREAKDOWN:
        return base
    return base + (
        f" {row['solve_s_total']:>9.3f} {row['n_iter']:>6d} {row['n_matvec']:>8d} "
        f"{row['t_matvec_total']:>11.3f} {row['t_matvec_avg']:>11.3e} {row['n_precond']:>8d} "
        f"{row['t_precond_total']:>13.3f} {row['t_precond_avg']:>12.3e} {row['t_cg_overhead']:>12.3f}"
    )

In [81]:
cache = {}
ref_cache = {}


def get_case_entry(n_value: int, sigma: float, eps: float, seed: int) -> dict:
    key = (int(n_value), float(sigma), float(eps))
    if key in cache:
        return cache[key]
    solver, state, pre_times = precompute_streaming_case(n_value, sigma, eps, seed)
    entry = {
        "N": int(n_value),
        "sigma": float(sigma),
        "eps": float(eps),
        "solver": solver,
        "state": state,
        "pre": float(pre_times["time_precompute_total"]),
    }
    cache[key] = entry
    return entry


def compute_reference_pred(n_value: int, sigma: float, seed: int, x_trg: np.ndarray) -> dict:
    key = (int(n_value), float(sigma), float(EPS_REF))
    if key in ref_cache:
        return ref_cache[key]
    solver, state, _pre_times = precompute_streaming_case(n_value, sigma, EPS_REF, seed)
    matvec = lambda v: solver._apply_A(state, v)
    beta, it, relres, _hist = bm.pcg_solve(
        matvec,
        state.rhs,
        tol=EPS_REF,
        maxiter=CG_MAXITER_REF,
        precond=None,
    )
    yhat_ref = solver.predict(x_trg, beta, state)
    entry = {
        "yhat_ref": yhat_ref,
        "iters": int(it),
        "relres": float(relres),
        "converged": bool(relres <= EPS_REF),
        "hit_maxiter": bool(it >= CG_MAXITER_REF),
    }
    ref_cache[key] = entry
    return entry


def solve_with_topq(entry: dict, top_q: int, x_trg: np.ndarray, y_trg_test: np.ndarray, ref=None) -> dict:
    solver = entry["solver"]
    state = entry["state"]
    eps = entry["eps"]
    matvec = lambda v: solver._apply_A(state, v)

    precond, _eigpairs, _mu, t_eig, t_precond, top_q_used = bm.build_precond_from_state(
        solver,
        state,
        top_q,
        eig_method=EIG_METHOD,
        eig_tol=EIG_TOL,
        eig_maxiter=EIG_MAXITER,
        eig_block_size=EIG_BLOCK_SIZE,
        eig_oversample=EIG_OVERSAMPLE,
    )
    precond_apply = precond.apply if precond is not None else None

    maxiter = CG_MAXITER_BASELINE if top_q_used == 0 else CG_MAXITER_PRECOND

    t_s0 = time.perf_counter()
    if ENABLE_SOLVE_BREAKDOWN:
        beta, it, relres, _hist, stats = bm.pcg_solve(
            matvec,
            state.rhs,
            tol=eps,
            maxiter=maxiter,
            precond=precond_apply,
            store_history=False,
            return_stats=True,
        )
    else:
        beta, it, relres, _hist = bm.pcg_solve(
            matvec,
            state.rhs,
            tol=eps,
            maxiter=maxiter,
            precond=precond_apply,
            store_history=False,
        )
        stats = None
    t_s1 = time.perf_counter()

    t_m0 = time.perf_counter()
    yhat = solver.predict(x_trg, beta, state)
    t_m1 = time.perf_counter()

    err_rms = rms(y_trg_test, yhat)
    if ref is None:
        err_eepm_new = float("nan")
        err_rmse_ex = float("nan")
        err_eepm_train = float("nan")
    else:
        yhat_ref = ref["yhat_ref"]
        err_eepm_new = rms(yhat, yhat_ref)
        err_rmse_ex = rms(y_trg_test, yhat_ref)
        compute_train = True
        if EEPM_TRAIN_TOPQ is not None and top_q_used not in EEPM_TRAIN_TOPQ:
            compute_train = False
        if EEPM_TRAIN_MAX_N is not None and entry["N"] > EEPM_TRAIN_MAX_N:
            compute_train = False
        if compute_train:
            err_eepm_train = compute_train_eepm(
                solver,
                state,
                beta,
                entry["N"],
                entry["sigma"],
                seed_for_case(entry["N"], SIGMA_MODE),
                max_n=EEPM_TRAIN_MAX_N,
            )
        else:
            err_eepm_train = float("nan")

    solve_time = t_s1 - t_s0
    mean_time = t_m1 - t_m0
    pre = entry["pre"]
    total = pre + t_eig + t_precond + solve_time + mean_time
    m = (state.grid.mtot - 1) // 2

    row = {
        "sigma": entry["sigma"],
        "N": entry["N"],
        "dim": int(KERNEL.dim),
        "kernel": KERNEL_LABEL,
        "eps": float(eps),
        "top_q": int(top_q_used),
        "m": int(m),
        "pre": float(pre),
        "eig": float(t_eig),
        "precond": float(t_precond),
        "solve": float(solve_time),
        "mean": float(mean_time),
        "total": float(total),
        "iters": int(it),
        "relres": float(relres),
        "converged": bool(relres <= eps),
        "hit_maxiter": bool(it >= maxiter),
        "eepm_train": float(err_eepm_train),
        "eepm_new": float(err_eepm_new),
        "rms": float(err_rms),
        "rmse_ex": float(err_rmse_ex),
    }

    if ENABLE_SOLVE_BREAKDOWN:
        n_matvec = stats["n_matvec"]
        t_matvec_total = stats["t_matvec_total"]
        n_precond = stats["n_precond"]
        t_precond_total = stats["t_precond_total"]
        t_matvec_avg = t_matvec_total / max(n_matvec, 1)
        t_precond_avg = t_precond_total / max(n_precond, 1)
        t_cg_overhead = solve_time - t_matvec_total - t_precond_total
        row.update(
            {
                "solve_s_total": float(solve_time),
                "n_iter": int(it),
                "n_matvec": int(n_matvec),
                "t_matvec_total": float(t_matvec_total),
                "t_matvec_avg": float(t_matvec_avg),
                "n_precond": int(n_precond),
                "t_precond_total": float(t_precond_total),
                "t_precond_avg": float(t_precond_avg),
                "t_cg_overhead": float(t_cg_overhead),
            }
        )

    return row


def _write_csv(rows: list[dict], path: Path):
    if not rows:
        return
    keys = list(rows[0].keys())
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def run_table(
    sigma_mode: str,
    n_list: list[int] | None = None,
    eps_list: list[float] | None = None,
    top_q_list: list[int] | None = None,
    print_rows: bool = True,
    max_rows: int | None = None,
    return_on_interrupt: bool = True,
    csv_path: Path | None = None,
):
    if n_list is None:
        n_list = list(N_LIST)
    if eps_list is None:
        eps_list = list(EPS_LIST)
    if top_q_list is None:
        top_q_list = list(TOP_Q_SWEEP)

    rows = []
    if print_rows:
        print_table_header()

    last_case = None
    try:
        for n_value in n_list:
            sigma = sigma_for_N(n_value, sigma_mode)
            seed = seed_for_case(n_value, sigma_mode)
            x_trg, y_trg_test, _y_true = make_targets(seed + 999, sigma)
            ref = compute_reference_pred(n_value, sigma, seed, x_trg) if COMPUTE_EEPM else None

            for eps in eps_list:
                entry = get_case_entry(n_value, sigma, eps, seed)
                for top_q in top_q_list:
                    last_case = (n_value, sigma, eps, top_q)
                    row = solve_with_topq(entry, top_q, x_trg, y_trg_test, ref=ref)
                    rows.append(row)
                    if print_rows:
                        print(format_table_row(row))
                    if max_rows is not None and len(rows) >= max_rows:
                        if csv_path is not None:
                            _write_csv(rows, csv_path)
                        return rows
    except KeyboardInterrupt:
        if return_on_interrupt:
            if last_case is not None:
                n_value, sigma, eps, top_q = last_case
                print(
                    "Interrupted at N={}, sigma={:.2e}, eps={:.1e}, top_q={}. Returning partial rows.".format(
                        n_value, sigma, eps, top_q
                    )
                )
            if csv_path is not None:
                _write_csv(rows, csv_path)
            return rows
        raise

    if csv_path is not None:
        _write_csv(rows, csv_path)
    return rows

In [82]:
# Table 3: fixed sigma
SIGMA_MODE = "table3"
rows_table3 = run_table(SIGMA_MODE, csv_path=CSV_PATH_TABLE3)

# Optional: make a DataFrame
# import pandas as pd
# df_table3 = pd.DataFrame(rows_table3)
# df_table3

sigma N d kernel eps top_q m pre_s eig_s precond_s solve_s mean_s total_s iters relres converged hit_max eepm_train eepm_new rms rmse_ex
1.00e-01    3000000 2   Mat32  1.0e-05     0   55    0.464    0.000    0.000    4.529    0.041    5.035   2155 9.64e-06         Y       N       nan       nan  1.00e-01       nan
1.00e-01    3000000 2   Mat32  1.0e-05    32   55    0.464    0.164    0.000    4.171    0.041    4.840   1058 9.76e-06         Y       N       nan       nan  1.00e-01       nan
1.00e-01    3000000 2   Mat32  1.0e-05    48   55    0.464    0.331    0.000    4.790    0.054    5.639    805 9.63e-06         Y       N       nan       nan  1.00e-01       nan
1.00e-01    3000000 2   Mat32  1.0e-05    64   55    0.464    0.666    0.000    5.808    0.060    6.999    619 9.70e-06         Y       N       nan       nan  1.00e-01       nan
1.00e-01    3000000 2   Mat32  1.0e-07     0  204    0.946    0.000    0.000  386.511    0.194  387.651   9062 8.59e-08         Y       N       nan    

In [ ]:
# Table 4: sigma scales with N
SIGMA_MODE = "table4"
rows_table4 = run_table(SIGMA_MODE, csv_path=CSV_PATH_TABLE4)

# Optional: make a DataFrame
# import pandas as pd
# df_table4 = pd.DataFrame(rows_table4)
# df_table4

sigma N d kernel eps top_q m pre_s eig_s precond_s solve_s mean_s total_s iters relres converged hit_max eepm_train eepm_new rms rmse_ex solve_s_total n_iter n_matvec t_matvec_total t_matvec_avg n_precond t_precond_total t_precond_avg t_cg_overhead
1.00e-01    3000000 2   Mat32  1.0e-05     0   55    0.509    0.000    0.000    4.744    0.044    5.298   2218 8.92e-06         Y       N       nan       nan  1.00e-01       nan     4.744   2218     2219       3.754   1.692e-03        0         0.000    0.000e+00        0.990
1.00e-01    3000000 2   Mat32  1.0e-05    16   55    0.509    0.115    0.000    4.150    0.045    4.820   1360 8.63e-06         Y       N       nan       nan  1.00e-01       nan     4.150   1360     1361       2.349   1.726e-03     1360         1.152    8.468e-04        0.650
1.00e-01    3000000 2   Mat32  1.0e-05    32   55    0.509    0.202    0.000    3.986    0.043    4.741   1042 9.70e-06         Y       N       nan       nan  1.00e-01       nan     3.986   1042   